In [1]:
import numpy as np
import pandas as pd
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import datetime
from pathlib import Path 
import re
import xarray as xr

In [2]:
from influpaint.datasets import mixer as dataset_mixer
from influpaint.utils import converters

Need for this file:
    - the imports above, with modifications in `converters.py` and `mixer.py` found in `adding-new-datasource` branch
    - the output from `prep_age_flu_hosp_data.ipynb` (at top level of `influpaint` directory, also in `adding-new-datasource` branch)
    - your desired output path for .nc NetCDF files (for final cell to run)

This file is an attempted modification of the logic found in influpaint/2-build_training_flu_datasets_ipynb.py

In [3]:
all_datasets_df = pd.read_csv(
    "/Users/emprzy/Documents/work/miscellaneous/influpaint_data/training_data.csv",
    dtype={
        'location_code': str,
        'value': np.float64,
        'fluseason_fraction': np.float64,
        'season_week': np.int64,
        'fluseason': np.int64,
        'datasetH1': str,
        'datasetH2': str,
        'sample': str  
    },
    parse_dates=['week_enddate']
)

In [17]:
age_groups = sorted(all_datasets_df['age_group'].unique())
# pivot the table to have one column per age group
df_wide = all_datasets_df.pivot_table(
    index=['location_code', 'season_week', 'week_enddate', 'sample', 'fluseason', 'datasetH1', 'datasetH2', 'fluseason_fraction'],
    columns='age_group',
    values='value'
).reset_index()
df_wide.columns.name = None

In [26]:
AGE_COLUMNS = [
        '0-130',
        '0-4',
        '18-49',
        '5-17',
        '50-64',
        '65-130'
    ]
build_frames_config = { 
    "NHSN": {"multiplier": 2080, "to_scale": False}, # creates 10k frames of training data, ~20% are surveillance
    "SMH_R4": {"multiplier": 1, "to_scale": False},
    "SMH_R5": {"multiplier": 1, "to_scale": False},
}
season_axis = set(all_datasets_df['location_code']) # my SeasonAxis replacement (PATCH)

In [27]:
frame_list = dataset_mixer.build_frames(
    df_wide, 
    build_frames_config,
    season_axis=season_axis, 
    fill_missing_locations="random",
) 

# version below is for the unpivoted data (i don't think this the right way to do this)
#frame_list = dataset_mixer.build_frames(
    #all_datasets_df, 
    #build_frames_config,
    #season_axis=season_axis, 
    #fill_missing_locations="random",
#) 

Pre-computing intelligent fill lookup table...
  Building location data lookup...
  Lookup table built for 52 locations
Building frames...
Processing NHSN (multiplier=2080)...


Processing SMH_R4 (multiplier=1)...


Processing SMH_R5 (multiplier=1)...


Created 10000 total frames:
  NHSN: 2080 frames (multiplier=2080)
  SMH_R4: 600 frames (multiplier=1)
  SMH_R5: 7320 frames (multiplier=1)


In [29]:
def build_dataset_from_framelist(frame_list, season_setup: set[str]):
    main_origins = []
    for i, frame in enumerate(frame_list):
        df = frame_list[i]
        df["fluseason"] = i  # Normalize sample ID
        frame_list[i] = df
        
        # Validation
        assert df.season_week.max() == 53 and df.season_week.min() == 1
        
        # Track Origin
        main_origins.append(df["origin"].mode()[0] if not df["origin"].mode().empty else None)

    all_frames_df = pd.concat(frame_list).reset_index(drop=True)
    
    # PATCHED to accommodate 6 channels
    array_list = converters.dataframe_to_arraylist( 
        df=all_frames_df, 
        locations=season_setup, 
        age_columns=AGE_COLUMNS
    )

    array = np.array(array_list) # Shape: (Samples, 6, 64, 64)

    flu_payload_array = xr.DataArray(
        array, 
        coords={
            'sample': np.arange(array.shape[0]),
            'feature': AGE_COLUMNS,                       # Coordinates for the 6 age groups
            'season_week_padded': np.arange(64),          # Time axis (0-63)
            'location_padded': np.arange(64)             # Spatial axis (0-63)
        }, 
        dims=["sample", "feature", "season_week_padded", "location_padded"]
    )
    return flu_payload_array, main_origins


In [30]:
flu_payload_array, main_origins = build_dataset_from_framelist(frame_list, season_setup=season_axis)

In [31]:
flu_payload_array.shape

(10000, 6, 64, 64)

In [32]:
flu_payload_array.to_netcdf("/Users/emprzy/Documents/work/miscellaneous/influpaint_data/flu_age_10k.nc")

Skipping visualization, since there are 6 channels 